## This notebook will deal with the following topics:
1. Partitioning
2. Liquid Clustering

### Liquid Clustering vs Partitioning (Brief)

#### Partitioning
- Splits the table into **separate folders**, one per distinct value of the partition column (e.g. `dt=2025-01-01/`).
- Data is routed to the correct partition **on every write**, so no extra job is needed to organize it.
- Queries filtering on the partition column **skip whole folders** (partition pruning).
- Best for **low-cardinality** columns (date, country) on very large tables.
- **Fixed at creation.** Changing the partition column requires rewriting the table.
- Over-partitioning creates **many small files** and slows queries.

#### Liquid Clustering
- Groups related rows **within files** by the clustering columns, with no folder-per-value layout.
- Works with **high-cardinality** columns (`id`, timestamps) as well as low-cardinality ones.
- **Flexible.** Change clustering columns anytime with `ALTER TABLE ... CLUSTER BY`, without a full rewrite.
- Clustering is applied when **`OPTIMIZE`** runs, so it acts like a **separate maintenance job** (schedule it, or let predictive optimization run it).
- New data stays unclustered until `OPTIMIZE` runs. Some write paths (e.g. CTAS, large inserts) may cluster on write, but it isn't guaranteed.
- `OPTIMIZE` is **incremental**: it only clusters new or unclustered data.
- Replaces **partitioning and ZORDER**. They can't be combined on the same table.

```sql
-- partitioning
CREATE TABLE t (id INT, dt DATE) PARTITIONED BY (dt);

-- liquid clustering
CREATE TABLE t (id INT, dt DATE) CLUSTER BY (dt, id);
ALTER TABLE t CLUSTER BY (id);   -- change later
OPTIMIZE t;                      -- apply clustering
```

#### Comparison
| | Partitioning | Liquid Clustering |
|---|---|---|
| Layout | Folders per value | Clustered rows within files |
| Best column type | Low cardinality | Any, incl. high cardinality |
| Data organized | On every write | When `OPTIMIZE` runs (separate job) |
| Change columns later | Requires table rewrite | `ALTER TABLE ... CLUSTER BY` |
| Small file risk | High if over-partitioned | Low |

#### Which to use
- **New tables:** liquid clustering (recommended by Databricks).
- **Existing partitioned tables / older runtimes:** partitioning with `ZORDER` still works.

In [0]:
%sql
CREATE OR REPLACE TABLE delta_table_practice.delta.sales_partitions_demo_partitions
(
    order_id    string,
    order_date  date,
    country     string,
    category    string,
    amount      double 
)
using delta
partitioned by (country, category)

In [0]:
%sql
INSERT INTO delta_table_practice.delta.sales_partitions_demo_partitions
VALUES
('1', '2022-01-01', 'USA', 'Electronics', 100),
('2', '2022-01-02', 'Canada', 'Clothing', 200),
('3', '2022-01-03', 'USA', 'Electronics', 300),
('4', '2022-01-04', 'Mexico', 'Electronics', 400),
('5', '2022-01-05', 'Canada', 'Clothing', 500)


In [0]:
df=spark.read.table("delta_table_practice.delta.sales_partitions_demo")
df.write.partitionBy("country","category").save("/Volumes/delta_table_practice/delta/delta_volume/delta_sales_partitioned")

In [0]:
%sql
CREATE OR REPLACE TABLE delta_table_practice.delta.sales_clustering_demo
(
    order_id    string,
    order_date  date,
    country     string,
    category    string,
    amount      double 
)
using delta
cluster by (country, category)

In [0]:
%sql
INSERT INTO delta_table_practice.delta.sales_clustering_demo
VALUES
('1', '2022-01-01', 'USA', 'Electronics', 100),
('2', '2022-01-02', 'Canada', 'Clothing', 200),
('3', '2022-01-03', 'USA', 'Electronics', 300),
('4', '2022-01-04', 'Mexico', 'Electronics', 400),
('5', '2022-01-05', 'Canada', 'Clothing', 500)
